# 实验零：实验概览与环境准备

**大规模训练加速与调优 | 1*NPU 910B3 + PASCAL VOC + AMP + MSPROF**

本实验的目标不是一次性追求多卡规模，而是在当前 `1*NPU 910B3、16 vCPU、32GiB 内存` 条件下，先把 YOLO 目标检测训练完整跑通，再用可观测的指标做加速与调优。

完成本实验后，你应该能够：

1. 判断 Ascend 训练环境是否可用，包括 Python、PyTorch、torch-npu、NPU 设备和 CANN 环境。
2. 使用 PASCAL VOC 作为目标检测数据集，并理解它为什么适合当前实验条件。
3. 在单卡 NPU 上启动 YOLO 训练，产生日志和 checkpoint。
4. 使用 AMP、Warmup、batch size、DataLoader workers 调整训练吞吐与稳定性。
5. 使用 MSPROF 定位 DataLoader gap、Host 调度和 NPU 算子耗时。
6. 理解 HCCL/DDP 如何从当前单卡脚本扩展到多卡，但本轮不强行运行多卡。


## 当前实验路线

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">阶段</th>
      <th style="text-align: left;">目标</th>
      <th style="text-align: left;">产出</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">1. 环境准备</td>
      <td style="text-align: left;">确认 Python、torch-npu、NPU 设备、磁盘空间</td>
      <td style="text-align: left;">环境检查记录</td>
    </tr>
    <tr>
      <td style="text-align: left;">2. 数据准备</td>
      <td style="text-align: left;">下载 VOC，转换 YOLO label，生成 split 列表</td>
      <td style="text-align: left;">可被 DataLoader 读取的数据集</td>
    </tr>
    <tr>
      <td style="text-align: left;">3. 单卡启动</td>
      <td style="text-align: left;">用 <code>launch_1npu.sh</code> 启动训练</td>
      <td style="text-align: left;">训练日志、checkpoint</td>
    </tr>
    <tr>
      <td style="text-align: left;">4. AMP 与 Warmup</td>
      <td style="text-align: left;">降低显存压力并稳定前期训练</td>
      <td style="text-align: left;">稳定的 loss 曲线</td>
    </tr>
    <tr>
      <td style="text-align: left;">5. 调参实验</td>
      <td style="text-align: left;">调整 batch size 与 <code>num_workers</code></td>
      <td style="text-align: left;">吞吐对比表</td>
    </tr>
    <tr>
      <td style="text-align: left;">6. MSPROF 分析</td>
      <td style="text-align: left;">采集 profiling 并定位瓶颈</td>
      <td style="text-align: left;">优化建议清单</td>
    </tr>
    <tr>
      <td style="text-align: left;">7. DDP 扩展</td>
      <td style="text-align: left;">理解多卡迁移方式</td>
      <td style="text-align: left;">后续多卡实验预留</td>
    </tr>
  </tbody>
</table>

`NPU` 是 Neural Processing Unit，即神经网络处理器。Ascend 910B3 是面向训练的 NPU，适合运行矩阵计算、卷积、归一化等深度学习算子。`torch-npu` 是 PyTorch 连接 Ascend NPU 的适配包，它让 `torch` 模型可以把张量和模型移动到 `npu:0` 上运行。


In [ ]:
# ====== 1. 检查 Python 与基础依赖 ======
import importlib
import platform
import sys

print('Python:', sys.version)
print('Platform:', platform.platform())

packages = ['torch', 'torchvision', 'torch_npu', 'yaml', 'PIL', 'numpy', 'tqdm']
for name in packages:
    try:
        module = importlib.import_module(name)
        version = getattr(module, '__version__', 'installed')
        print(f'{name:12s}: {version}')
    except Exception as exc:
        print(f'{name:12s}: missing or failed to import -> {exc}')


In [ ]:
# ====== 2. 检查 NPU 是否可见 ======
import subprocess

try:
    import torch
    import torch_npu  # noqa: F401
    print('torch.npu.is_available():', torch.npu.is_available())
    print('torch.npu.device_count():', torch.npu.device_count())
    if torch.npu.is_available():
        print('current NPU:', torch.npu.current_device())
except Exception as exc:
    print('torch-npu 检查失败:', exc)

try:
    result = subprocess.run(['npu-smi', 'info'], text=True, capture_output=True, timeout=10)
    print(result.stdout or result.stderr)
except Exception as exc:
    print('npu-smi 不可用或执行失败:', exc)


## 数据和代码应该放在哪里

训练数据不要放进 git 仓库。VOC 解压后虽然比 COCO 小很多，但仍然有数 GB，推荐放在 `/mnt/workspace/datasets/voc`。

`/mnt/workspace` 通常是训练平台给你的工作盘。它和系统目录不同：系统 Python、CANN、驱动等可能放在 `/usr/local` 或 `/opt`，而你自己的代码、数据和 checkpoint 更适合放在 `/mnt/workspace`。


In [ ]:
# ====== 3. 检查 /mnt/workspace 是否适合放 PASCAL VOC ======
import shutil
from pathlib import Path

workspace = Path('/mnt/workspace')
if workspace.exists():
    total, used, free = shutil.disk_usage(workspace)
    gb = 1024 ** 3
    print(f'/mnt/workspace total: {total / gb:.1f} GB')
    print(f'/mnt/workspace used : {used / gb:.1f} GB')
    print(f'/mnt/workspace free : {free / gb:.1f} GB')
    print('建议 VOC 放在 /mnt/workspace/datasets/voc')
else:
    print('/mnt/workspace 不存在，请根据服务器实际挂载点调整 src/configs/yolo_ascend.yaml 中 data.root')


## 本章小结

当前实验路线已经明确：先用 PASCAL VOC 完成单卡 NPU 训练闭环，再对单卡吞吐和瓶颈做调优。下一章进入数据下载、XML 标注转换和 YOLO 配置检查。


## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) 本实验的主线目标最准确的是哪一项？
   - A. 从零训练 ResNet 分类模型
   - B. 在 Ascend NPU 上完成 YOLO 单卡训练流程，并了解多卡扩展入口
   - C. 只学习 Git LFS 的使用
   - D. 只完成数据可视化

2. (单选题) 确认 NPU 是否被系统识别，优先使用的命令是？
   - A. npu-smi info
   - B. git status
   - C. python --version
   - D. lsblk

3. (单选题) 本实验中 Python 与 torch_npu 的主要作用是？
   - A. 管理 Git 分支
   - B. 让 PyTorch 训练代码能够调用 Ascend NPU
   - C. 把 XML 文件压缩成 zip
   - D. 替代 CANN Toolkit

4. (单选题) 下列哪一项最适合作为实验输出是否可信的判断依据？
   - A. 只看命令能否启动
   - B. 只看终端颜色是否为绿色
   - C. 同时检查 loss、吞吐、checkpoint 和关键日志
   - D. 只看文件夹是否存在

5. (多选题) 进入正式训练前，通常需要检查哪些内容？
   - A. CANN/Ascend 环境变量
   - B. Python 依赖和 torch_npu 可用性
   - C. 数据集目录与配置路径
   - D. 是否已经安装 Office 插件

6. (多选题) 本实验涉及的主要文件类型包括哪些？
   - A. Jupyter Notebook
   - B. YAML 配置文件
   - C. Python 训练脚本
   - D. PASCAL VOC 标注 XML/YOLO txt

7. (多选题) 下列哪些现象通常说明 Ascend NPU 训练环境还没有准备好？
   - A. import torch_npu 失败
   - B. npu-smi info 看不到可用设备
   - C. 训练脚本提示找不到数据集路径
   - D. README 能正常打开

8. (判断题) 本实验默认要求必须完成多卡 DDP 训练，否则实验不完整。

9. (判断题) Notebook 中每一节的运行结果都应尽量保留关键输出，便于复现实验过程。

10. (填空题) Ascend NPU 的 PyTorch 适配通常依赖 Python 包 `____`。

11. (填空题) 本实验默认训练设备是单张 Ascend NPU，因此启动脚本通常只需要设置 device 为 `____`。

12. (简答题) 为什么本实验先跑通单卡流程，再把多卡 DDP 放到扩展部分？

13. (简答题) 实验报告中描述 NPU 训练是否成功时，应至少包含哪些证据？

14. (简答题) 如果 notebook 中命令在终端能跑、但在 Jupyter 中失败，排查思路是什么？

15. (代码设计题) 写一段最小 Python 检查代码，打印 PyTorch、torch_npu 是否可导入，并输出 NPU 是否可用。

> 参考答案见 answer/03.01_chapter_intro_answer.ipynb。
